### We are using the leNet5 CNN for this

2 feature exrtractor CNN with LeNet5, with 35 epochs and on the test/train split created we get a decent result of around 0.9985 in terms of accuracy however it does overfit to the data very easily. Maybe some augumentation could fix this (dont let the loss function reach 0.00)




In [63]:
from pathlib import Path
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from scipy.ndimage import center_of_mass, shift


In [64]:
TrainingDirectory = Path("iivp-2026-challenge/train/train")
X_train, y_train = [], []
for class_dir in sorted(TrainingDirectory.iterdir(), key=lambda p: int(p.name)):
    label = int(class_dir.name)
    for png in class_dir.glob("*.png"):
        X_train.append(np.array(Image.open(png), dtype=np.uint8))
        y_train.append(label)

X_train = np.stack(X_train)
y_train = np.array(y_train)

print(X_train.shape, y_train.shape)

def recenter(img):
    cy,cx = center_of_mass(img)
    if np.isnan(cy):
        return img
    return shift(img, (img.shape[0]/2 - cy, img.shape[1]/2 -cx), order=1)

X_train_centered = np.array([recenter(img) for img in X_train])


(17000, 32, 32) (17000,)


In [65]:
class LeNet5(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1 = nn.Conv2d(1,6, kernel_size=5)
        self.pool = nn.MaxPool2d(2,2)
        self.conv2 = nn.Conv2d(6,16,kernel_size=5)
        self.fc1 = nn.Linear(16*5*5, 120)
        self.fc2 = nn.Linear(120,84)
        self.fc3 = nn.Linear(84, num_classes)


    def forward(self,x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = torch.flatten(x,1)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

In [66]:
model = LeNet5()

In [67]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_centered, y_train,
    test_size=0.2, stratify=y_train, random_state=42,
)

X_tr_t  = torch.tensor(X_tr,  dtype=torch.float32).unsqueeze(1) / 255.0
X_val_t = torch.tensor(X_val, dtype=torch.float32).unsqueeze(1) / 255.0
y_tr_t  = torch.tensor(y_tr,  dtype=torch.long)
y_val_t = torch.tensor(y_val, dtype=torch.long)

train_ds = TensorDataset(X_tr_t, y_tr_t)
val_ds   = TensorDataset(X_val_t, y_val_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=256, shuffle=False)

print(X_tr_t.shape, y_tr_t.shape)


torch.Size([13600, 1, 32, 32]) torch.Size([13600])


In [68]:
dev = "mps"
model = LeNet5().to(dev)
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)
criterion = nn.CrossEntropyLoss()

In [69]:
def evaluate(loader):
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for xd, yd in loader:
            xd, yd = xd.to(dev), yd.to(dev)
            pred = model(xd).argmax(dim=1)
            correct += (pred == yd).sum().item()
            total += yd.size(0)
    return correct / total

epochs = 35
for epoch in range(1, epochs + 1):
    model.train()
    run_loss = 0.0
    for xd, yd in train_loader:
        xd, yd = xd.to(dev), yd.to(dev)
        optimizer.zero_grad()
        loss = criterion(model(xd),yd)
        loss.backward()
        optimizer.step()
        run_loss += loss.item() * xd.size(0)

    train_loss = run_loss / len(train_ds)
    val_acc = evaluate(val_loader)
    print(f"epoch {epoch:2d}  loss {train_loss:.4f}  val_acc {val_acc:.4f}")

    

epoch  1  loss 0.6203  val_acc 0.9362
epoch  2  loss 0.1401  val_acc 0.9653
epoch  3  loss 0.0804  val_acc 0.9829
epoch  4  loss 0.0532  val_acc 0.9844
epoch  5  loss 0.0369  val_acc 0.9882
epoch  6  loss 0.0325  val_acc 0.9832
epoch  7  loss 0.0259  val_acc 0.9885
epoch  8  loss 0.0211  val_acc 0.9862
epoch  9  loss 0.0162  val_acc 0.9918
epoch 10  loss 0.0137  val_acc 0.9932
epoch 11  loss 0.0096  val_acc 0.9915
epoch 12  loss 0.0067  val_acc 0.9947
epoch 13  loss 0.0177  val_acc 0.9929
epoch 14  loss 0.0060  val_acc 0.9941
epoch 15  loss 0.0108  val_acc 0.9935
epoch 16  loss 0.0076  val_acc 0.9929
epoch 17  loss 0.0032  val_acc 0.9938
epoch 18  loss 0.0094  val_acc 0.9847
epoch 19  loss 0.0071  val_acc 0.9932
epoch 20  loss 0.0027  val_acc 0.9950
epoch 21  loss 0.0046  val_acc 0.9950
epoch 22  loss 0.0107  val_acc 0.9918
epoch 23  loss 0.0052  val_acc 0.9894
epoch 24  loss 0.0032  val_acc 0.9924
epoch 25  loss 0.0017  val_acc 0.9944
epoch 26  loss 0.0011  val_acc 0.9956
epoch 27  lo